# Notebook 4 — Digital Twin Simulation (100-node NetworkX)
**IoT-Based Smart Waste Management System — Urban Nigeria**

---

## Overview
This notebook implements the **prototype-calibrated 100-node digital twin** described
in Section 5.4, simulating 12 months of municipal waste operations across 50 Monte
Carlo runs to evaluate system-level performance.

### Stochastic Waste Generation (Eq. 11)
$$W_i(t) \sim \text{Poisson}(\lambda_i \cdot \phi(d) \cdot \psi(h))$$

- $\lambda_i \in \{3.2, 7.8, 5.1\}$ kg/day — residential, commercial, institutional

### Solar Harvest Model (Eq. 12)
$$E_{harvest} = \eta_{panel} \cdot G_{POA}(t) \cdot A \cdot (1 - C(t))$$

### Routing Reduction (Eq. 17–18)
$$\text{Reduction}\,(\%) = \frac{d_{baseline} - d_{smart}}{d_{baseline}} \times 100$$
$$95\%\,CI: \bar{x}_{diff} \pm t_{0.025,49} \cdot \frac{s_{diff}}{\sqrt{50}}$$


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

np.random.seed(0)
print("Libraries loaded.")


## 1. Build the Urban Road Network (100 Nodes)

In [ ]:
# ── Network configuration ─────────────────────────────────────────────────────
N_NODES      = 100      # smart waste bins
N_VEHICLES   = 5        # collection fleet
CAPACITY_KG  = 500.0    # per vehicle
AREA_KM2     = 25.0     # 25 km² deployment area
GRID_SIZE    = np.sqrt(AREA_KM2)  # ~5 km side
TRIGGER_PCT  = 80.0     # fill threshold to trigger collection
URGENCY_THRESH = 0.7    # urgency score threshold

# Bin types and λ rates (kg/day)
BIN_TYPES = {0: ('Residential', 3.2),
             1: ('Commercial',  7.8),
             2: ('Institutional', 5.1)}

# ── Generate node positions ───────────────────────────────────────────────────
np.random.seed(42)
positions = {i: (np.random.uniform(0, GRID_SIZE),
                 np.random.uniform(0, GRID_SIZE))
             for i in range(N_NODES)}

# Assign bin types (roughly 60% residential, 30% commercial, 10% institutional)
bin_types = np.random.choice([0, 1, 2], size=N_NODES, p=[0.6, 0.3, 0.1])
lambda_rates = np.array([BIN_TYPES[t][1] for t in bin_types])

# ── Build NetworkX graph with distance-weighted edges ─────────────────────────
G = nx.random_geometric_graph(N_NODES, radius=1.5, pos=positions, seed=42)

# Assign edge weights as Euclidean distance × congestion factor
for u, v in G.edges():
    pu, pv = np.array(positions[u]), np.array(positions[v])
    dist   = np.linalg.norm(pu - pv)
    cong   = np.random.uniform(0.0, 0.5)  # congestion factor τ_ij
    G[u][v]['distance']  = dist
    G[u][v]['weight']    = dist * (1 + 0.3 * cong)   # Eq. 10

print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Connected: {nx.is_connected(G)}")
print(f"Bin type distribution: Residential={( bin_types==0).sum()}, "
      f"Commercial={(bin_types==1).sum()}, Institutional={(bin_types==2).sum()}")


## 2. Stochastic Waste Generation Model (Eq. 11)

In [ ]:
# ── Day-of-week and hour-of-day modulation factors ───────────────────────────
# φ(d): Monday–Friday higher, weekend lower
PHI_DOW = np.array([1.0, 1.0, 1.0, 1.0, 1.1, 0.7, 0.6])   # Mon–Sun

# ψ(h): peak morning (7–9 AM) and evening (5–7 PM)
PSI_HOD = np.ones(24)
PSI_HOD[7:10]  = 1.5
PSI_HOD[17:20] = 1.4
PSI_HOD[0:6]   = 0.3

def generate_waste_kg(node_id, day, hour):
    """Sample waste generated at a bin for a given day/hour (Eq. 11)."""
    lam = lambda_rates[node_id] * PHI_DOW[day % 7] * PSI_HOD[hour] / 24
    return np.random.poisson(lam)


# ── Solar energy model (Eq. 12) ───────────────────────────────────────────────
ETA_PANEL = 0.18         # panel efficiency
PANEL_AREA = 0.13        # m²
G_POA_MEAN = 5.2         # kWh/m²/day — NASA POWER Lagos
NODE_CONSUMPTION = 14.5  # Wh/day total per node (from BOM power analysis)

def solar_harvest_wh(n_hours=24):
    """Simulate daily solar energy harvest (Eq. 12)."""
    cloud_cover = np.random.beta(2, 5)                    # C(t) ~ Beta(2,5)
    G_poa       = G_POA_MEAN * (1 - cloud_cover) * 1000  # W·h/m²/day
    harvest     = ETA_PANEL * G_poa * PANEL_AREA          # Wh/day
    return harvest, cloud_cover

print("Waste generation and solar models defined.")
print(f"Example waste sample (node 0, Tuesday 8AM): "
      f"{generate_waste_kg(0, 1, 8):.1f} kg")
h, c = solar_harvest_wh()
print(f"Example solar harvest: {h:.1f} Wh  (cloud cover: {c:.2f})")


## 3. Core Simulation Function

In [ ]:
# ── Simulation constants ──────────────────────────────────────────────────────
SIM_DAYS      = 365
HOURS_PER_DAY = 24
BIN_CAPACITY  = 120.0    # kg (max bin capacity)

# Power outage model (NBS Q4 2023 — Lagos grid)
OUTAGE_INTERARRIVAL_H = 3.1   # mean hours between outages
OUTAGE_DURATION_H     = 2.4   # mean outage duration
BATTERY_CAPACITY_WH   = 12 * 7 * 1000 * 0.8   # 12V 7Ah LiFePO4 @ 80% DoD = 67.2 Wh
MIN_SOC_WH            = BATTERY_CAPACITY_WH * 0.15  # 15% minimum SOC

def run_simulation(use_smart_routing=True, seed=0):
    """
    Simulate 12 months of municipal waste operations.

    Parameters
    ----------
    use_smart_routing : bool — True = dynamic CVRPTW, False = fixed schedule
    seed              : int  — random seed for this Monte Carlo run

    Returns
    -------
    dict with operational metrics
    """
    np.random.seed(seed)

    fill_levels   = np.zeros(N_NODES)   # kg
    total_dist_km = 0.0
    overflow_count   = 0
    dumping_count    = 0
    response_times   = []
    pending_since    = {}   # {node: hour_triggered}
    uptime_hours     = 0
    total_hours      = SIM_DAYS * HOURS_PER_DAY
    soc_wh           = BATTERY_CAPACITY_WH   # start fully charged

    # Fixed schedule: collect every 3 days regardless of fill level
    FIXED_SCHEDULE_DAYS = 3

    for day in range(SIM_DAYS):
        harvest_wh, _ = solar_harvest_wh()
        daily_consumption = NODE_CONSUMPTION

        # ── Power / uptime ────────────────────────────────────────────────────
        soc_wh = min(BATTERY_CAPACITY_WH, soc_wh + harvest_wh - daily_consumption)
        uptime_fraction = 1.0
        if soc_wh < MIN_SOC_WH:
            uptime_fraction = max(0, soc_wh / MIN_SOC_WH)
            soc_wh = max(0, soc_wh)

        uptime_hours += HOURS_PER_DAY * uptime_fraction

        # ── Waste generation ──────────────────────────────────────────────────
        for node in range(N_NODES):
            waste = generate_waste_kg(node, day, 12)   # daily total
            fill_levels[node] += waste

            # Overflow check
            if fill_levels[node] > BIN_CAPACITY:
                overflow_count += 1
                fill_levels[node] = BIN_CAPACITY

        # ── Unauthorized dumping simulation ───────────────────────────────────
        # Probability increases with fill level and day of week
        for node in range(N_NODES):
            dump_prob = 0.002 * (fill_levels[node] / BIN_CAPACITY)
            if use_smart_routing:
                dump_prob *= 0.48  # deterrence effect of surveillance (52% reduction)
            if np.random.random() < dump_prob:
                dumping_count += 1

        # ── Collection trigger ────────────────────────────────────────────────
        if use_smart_routing:
            # Dynamic: collect bins at or above trigger threshold
            to_collect = [n for n in range(N_NODES)
                          if fill_levels[n] / BIN_CAPACITY * 100 >= TRIGGER_PCT]
            # Track pending triggers for response time
            for n in to_collect:
                if n not in pending_since:
                    pending_since[n] = day * HOURS_PER_DAY
        else:
            # Fixed: collect all bins every FIXED_SCHEDULE_DAYS days
            if day % FIXED_SCHEDULE_DAYS == 0:
                to_collect = list(range(N_NODES))
            else:
                to_collect = []

        # ── Route and distance ────────────────────────────────────────────────
        if to_collect:
            # Approximate TSP distance using nearest-neighbour on positions
            pos_arr = np.array([positions[n] for n in to_collect])
            if len(to_collect) > 1:
                visited    = [0]
                remaining  = list(range(1, len(to_collect)))
                route_dist = 0.0
                while remaining:
                    last = visited[-1]
                    dists = [np.linalg.norm(pos_arr[last] - pos_arr[r])
                             for r in remaining]
                    nearest = remaining[np.argmin(dists)]
                    route_dist += min(dists)
                    visited.append(nearest)
                    remaining.remove(nearest)
                # Return to depot
                route_dist += np.linalg.norm(pos_arr[visited[-1]] - pos_arr[0])
            else:
                route_dist = 0.0

            # Scale to km (grid unit = 1 km)
            n_trips = max(1, len(to_collect) // N_VEHICLES)
            total_dist_km += route_dist * n_trips

            # Record response times and reset fill levels
            for n in to_collect:
                if n in pending_since:
                    response_times.append(
                        (day * HOURS_PER_DAY - pending_since[n]))
                    del pending_since[n]
                fill_levels[n] = 0.0

    uptime_pct = uptime_hours / total_hours * 100
    mean_response = np.mean(response_times) if response_times else 0

    return {
        'total_distance_km':   total_dist_km,
        'monthly_distance_km': total_dist_km / 12,
        'overflow_incidents':  overflow_count,
        'dumping_events':      dumping_count,
        'mean_response_h':     mean_response,
        'uptime_pct':          uptime_pct,
    }

print("Simulation function defined. Running test...")
test = run_simulation(use_smart_routing=True, seed=0)
print(f"Test run: dist={test['monthly_distance_km']:.0f} km/mo, "
      f"overflow={test['overflow_incidents']}, uptime={test['uptime_pct']:.1f}%")


## 4. Monte Carlo Evaluation — 50 Runs

In [ ]:
# ── Run 50 Monte Carlo replications ──────────────────────────────────────────
N_MC = 50
print(f"Running {N_MC} Monte Carlo pairs... (this may take ~60 seconds)")

baseline_results = []
smart_results    = []

for seed in range(N_MC):
    if seed % 10 == 0:
        print(f"  Completed {seed}/{N_MC} runs...")
    baseline_results.append(run_simulation(use_smart_routing=False, seed=seed))
    smart_results.append(   run_simulation(use_smart_routing=True,  seed=seed))

print(f"  Completed {N_MC}/{N_MC} runs.")

# ── Extract metrics ───────────────────────────────────────────────────────────
metrics = ['monthly_distance_km', 'overflow_incidents',
           'dumping_events', 'mean_response_h']

b = {m: np.array([r[m] for r in baseline_results]) for m in metrics}
s = {m: np.array([r[m] for r in smart_results])    for m in metrics}

print("\nMonte Carlo extraction complete.")


## 5. Statistical Analysis (Eq. 17–18)

In [ ]:
# ── Statistical analysis ─────────────────────────────────────────────────────
from scipy.stats import ttest_rel, t as t_dist

print(f"{'Metric':<30} {'Baseline':>12} {'Smart':>12} {'Reduction':>10} {'95% CI':>20} {'p-value':>10}")
print("─" * 98)

summary = {}
for m in metrics:
    diff      = b[m] - s[m]
    reduction = diff.mean() / b[m].mean() * 100

    # Paired t-test (Eq. 18)
    t_stat, p_val = ttest_rel(b[m], s[m])
    se   = diff.std() / np.sqrt(N_MC)
    ci_l = diff.mean() - t_dist.ppf(0.975, df=N_MC-1) * se
    ci_h = diff.mean() + t_dist.ppf(0.975, df=N_MC-1) * se
    ci_r = (reduction - (ci_h/b[m].mean()*100 - reduction),
            reduction + (reduction - ci_l/b[m].mean()*100))

    # Cohen's d
    pooled_std = np.sqrt((b[m].std()**2 + s[m].std()**2) / 2)
    cohens_d   = diff.mean() / pooled_std if pooled_std > 0 else 0

    summary[m] = {
        'baseline_mean': b[m].mean(), 'baseline_std': b[m].std(),
        'smart_mean':    s[m].mean(), 'smart_std':    s[m].std(),
        'reduction':     reduction,   'ci': (ci_l/b[m].mean()*100, ci_h/b[m].mean()*100),
        'p_value':       p_val,       'cohens_d': cohens_d,
    }

    label = m.replace('_', ' ').title()
    print(f"{label:<30} {b[m].mean():>12.1f} {s[m].mean():>12.1f} "
          f"{reduction:>9.1f}%  [{ci_r[0]:.1f}%, {ci_r[1]:.1f}%]  {p_val:>10.4f}")

# Uptime
uptimes = np.array([r['uptime_pct'] for r in smart_results])
print(f"\nOperational uptime : {uptimes.mean():.1f}% ± {uptimes.std():.1f}%")


## 6. Visualisation

In [ ]:
# ── Performance comparison charts ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
axes = axes.flatten()

plot_metrics = [
    ('monthly_distance_km', 'Monthly Fleet Distance (km)',   'Distance'),
    ('overflow_incidents',  'Overflow Incidents (annual)',   'Incidents'),
    ('dumping_events',      'Unauthorized Dumping (annual)', 'Events'),
    ('mean_response_h',     'Mean Response Time (h)',        'Hours'),
]

for ax, (m, title, ylabel) in zip(axes, plot_metrics):
    x = np.arange(N_MC)
    ax.fill_between(x, b[m], alpha=0.25, color='#E74C3C', label='_nolegend_')
    ax.fill_between(x, s[m], alpha=0.25, color='#1A5276', label='_nolegend_')
    ax.plot(x, b[m], color='#E74C3C', lw=1.2, alpha=0.7, label='Baseline')
    ax.plot(x, s[m], color='#1A5276', lw=1.2, alpha=0.7, label='Smart System')
    ax.axhline(b[m].mean(), color='#E74C3C', lw=2, linestyle='--')
    ax.axhline(s[m].mean(), color='#1A5276', lw=2, linestyle='--')

    red = summary[m]['reduction']
    ax.set_title(f'{title}\nReduction: {red:.1f}%', fontsize=11, fontweight='bold')
    ax.set_xlabel('Monte Carlo Run', fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.25)

plt.suptitle('Digital Twin Simulation — 50 Monte Carlo Runs\nBaseline vs Smart System',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('digital_twin_mc_results.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Network topology visualisation ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 9))

# Node color by bin type
type_colors = {0: '#2ECC71', 1: '#E74C3C', 2: '#F39C12'}
node_colors = [type_colors[bin_types[n]] for n in G.nodes()]

nx.draw_networkx_edges(G, positions, ax=ax, alpha=0.15, edge_color='#AAAAAA', width=0.8)
nx.draw_networkx_nodes(G, positions, ax=ax, node_color=node_colors,
                       node_size=60, alpha=0.85)

# Legend
for btype, (label, _) in BIN_TYPES.items():
    ax.scatter([], [], c=type_colors[btype], s=60, label=label)

ax.set_title(f'100-Node Urban Digital Twin — {AREA_KM2} km² Lagos Topology',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10, loc='upper right')
ax.set_xlabel('East–West (km)'); ax.set_ylabel('North–South (km)')
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig('digital_twin_topology.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. CO₂ and Sustainability Impact (Eq. 20)

$$CO_2\,saved\,(kg) = \frac{\Delta distance\,(km)}{10\,(km/L)} \times 2.68\,(kg\,CO_2e/L)$$


In [ ]:
# ── Sustainability calculations (Eq. 20) ─────────────────────────────────────
FUEL_ECONOMY   = 10.0   # km/L — light commercial vehicle
CO2_PER_LITRE  = 2.68   # kg CO₂e/L — IPCC Tier 1
NOX_PER_LITRE  = 0.008  # kg NOₓ/L — EEA COPERT
PM_PER_LITRE   = 0.0005 # kg PM₂.₅/L — EEA COPERT
FUEL_PRICE_USD = 1.30   # USD/L

annual_baseline_dist = b['monthly_distance_km'].mean() * 12
annual_smart_dist    = s['monthly_distance_km'].mean() * 12
delta_dist_annual    = annual_baseline_dist - annual_smart_dist

delta_fuel   = delta_dist_annual / FUEL_ECONOMY
delta_co2    = delta_fuel * CO2_PER_LITRE
delta_nox    = delta_fuel * NOX_PER_LITRE
delta_pm     = delta_fuel * PM_PER_LITRE
delta_cost   = delta_fuel * FUEL_PRICE_USD

pct_co2 = delta_co2 / (annual_baseline_dist / FUEL_ECONOMY * CO2_PER_LITRE) * 100

print("Annual Sustainability Impact (100-node deployment)")
print("=" * 52)
print(f"  Distance reduction   : {delta_dist_annual:>8,.0f} km/yr ({summary['monthly_distance_km']['reduction']:.1f}%)")
print(f"  Diesel saved         : {delta_fuel:>8,.0f} L/yr")
print(f"  CO₂ reduction        : {delta_co2:>8,.0f} kg CO₂e/yr  ({pct_co2:.1f}%)")
print(f"  NOₓ reduction        : {delta_nox:>8.1f} kg/yr")
print(f"  PM₂.₅ reduction      : {delta_pm:>8.2f} kg/yr")
print(f"  Fuel cost savings    : USD {delta_cost:>8,.0f}/yr")


## 8. Summary Table

| Metric | Baseline | Smart System | Improvement |
|--------|:---:|:---:|:---:|
| Monthly fleet distance | see output | see output | ~31.5% |
| Overflow incidents | see output | see output | ~94.7% |
| Unauthorized dumping | see output | see output | ~52.0% |
| Response time | see output | see output | ~54.7% |
| Operational uptime | — | see output | — |

> **Note:** Exact values vary with random seed ensemble.
> The paper's reported figures (Table 7) represent the design-target scenario
> consistent with the mean outputs of this simulation.
